# Análisis exploratorio, estadística descriptiva y visualización de patrones

## Curso de Analítica de Datos

Este notebook desarrolla de forma teórica, matemática y computacional los fundamentos del **Análisis Exploratorio de Datos (EDA)**, la **estadística descriptiva** y la **visualización de patrones**.

Se utilizan:

- **NumPy** para cálculo numérico.
- **Pandas** para manipulación de datos.
- **Matplotlib** para visualización básica.
- **Seaborn** para visualización estadística.

---

## Objetivos de aprendizaje

Al finalizar, el estudiante podrá:

1. Explicar el propósito del análisis exploratorio de datos.
2. Identificar la unidad de observación y clasificar variables.
3. Evaluar la calidad de una base de datos.
4. Calcular e interpretar medidas de tendencia central.
5. Calcular e interpretar medidas de dispersión y posición.
6. Analizar la forma de una distribución.
7. Detectar valores atípicos mediante reglas estadísticas.
8. Estudiar asociaciones entre variables cuantitativas y cualitativas.
9. Construir gráficos adecuados para cada tipo de variable.
10. Identificar patrones sin confundir asociación con causalidad.
11. Comunicar resultados mediante conclusiones sustentadas.

# 1. Preparación del entorno

Ejecute la siguiente celda para importar las librerías necesarias.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

np.random.seed(42)
sns.set_theme(style="whitegrid")

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Entorno preparado correctamente.")
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Seaborn:", sns.__version__)

# 2. ¿Qué es el análisis exploratorio de datos?

El **Análisis Exploratorio de Datos**, o **EDA** por sus siglas en inglés, es el conjunto de procedimientos utilizados para comprender una base antes de construir modelos estadísticos o tomar decisiones.

El EDA busca responder preguntas como:

- ¿Cuántas observaciones existen?
- ¿Qué representa cada fila?
- ¿Qué representa cada columna?
- ¿Qué tipo de variable es cada columna?
- ¿Hay valores faltantes?
- ¿Existen registros duplicados?
- ¿Hay valores imposibles?
- ¿Cómo se distribuyen las variables?
- ¿Existen valores atípicos?
- ¿Qué asociaciones aparecen?
- ¿Qué patrones merecen investigación?

El EDA combina tres componentes:

\[
\boxed{
\text{EDA}
=
\text{inspección}
+
\text{estadística descriptiva}
+
\text{visualización}
}
\]

No consiste únicamente en producir gráficos. También exige justificar cada cálculo e interpretar los resultados.

## 2.1 Población, muestra, observación y variable

### Población

Es el conjunto completo de elementos sobre los cuales se desea estudiar una característica.

### Muestra

Es un subconjunto de la población seleccionado para realizar el análisis.

### Observación

Es una unidad individual registrada en la base. Normalmente corresponde a una fila.

### Variable

Es una característica medida sobre cada observación. Normalmente corresponde a una columna.

Si la base tiene \(n\) observaciones y \(p\) variables, puede representarse mediante una matriz:

\[
\mathbf{X}
=
\begin{pmatrix}
x_{11} & x_{12} & \cdots & x_{1p}\\
x_{21} & x_{22} & \cdots & x_{2p}\\
\vdots & \vdots & \ddots & \vdots\\
x_{n1} & x_{n2} & \cdots & x_{np}
\end{pmatrix}.
\]

Aquí:

- \(x_{ij}\) representa el valor de la variable \(j\) en la observación \(i\);
- \(n\) es el número de filas;
- \(p\) es el número de columnas.

## 2.2 Tipos de variables

### Variables cualitativas

Describen categorías o atributos.

- **Nominales:** las categorías no tienen orden natural.
- **Ordinales:** las categorías tienen un orden natural.

### Variables cuantitativas

Representan cantidades numéricas.

- **Discretas:** generalmente provienen de conteos.
- **Continuas:** pueden tomar valores dentro de un intervalo.

### Identificadores

Un identificador puede estar compuesto por números, pero no representa una magnitud matemática. Por esta razón, no debe analizarse como una variable cuantitativa ordinaria.

# 3. Base de datos para el estudio

Se construirá una base sintética de clientes. La base contiene patrones deliberados para que puedan analizarse relaciones reales entre variables.

La unidad de observación es:

\[
\boxed{\text{un cliente}}
\]

In [ ]:
n = 400

clientes = pd.DataFrame({
    "cliente_id": np.arange(1001, 1001 + n),

    "edad": np.random.normal(
        loc=38,
        scale=12,
        size=n
    ).round(),

    "genero": np.random.choice(
        ["Femenino", "Masculino", "Otro"],
        size=n,
        p=[0.49, 0.48, 0.03]
    ),

    "region": np.random.choice(
        ["Norte", "Centro", "Sur", "Oriente"],
        size=n,
        p=[0.25, 0.35, 0.25, 0.15]
    ),

    "segmento": np.random.choice(
        ["Básico", "Intermedio", "Premium"],
        size=n,
        p=[0.45, 0.35, 0.20]
    ),

    "ingreso_mensual": np.random.lognormal(
        mean=np.log(2800),
        sigma=0.45,
        size=n
    ).round(2),

    "visitas_mes": np.random.poisson(
        lam=5,
        size=n
    ),

    "satisfaccion": np.random.normal(
        loc=4.0,
        scale=0.65,
        size=n
    ).clip(1, 5).round(1)
})

clientes.head()

## 3.1 Construcción de patrones

Se introduce una relación entre:

- ingreso y gasto;
- visitas y compras;
- satisfacción y abandono;
- segmento y gasto.

Esto permite estudiar patrones mediante gráficos y medidas de asociación.

In [ ]:
ajuste_segmento = clientes["segmento"].map({
    "Básico": 0,
    "Intermedio": 300,
    "Premium": 850
})

clientes["gasto_mensual"] = (
    0.28 * clientes["ingreso_mensual"]
    + 95 * clientes["visitas_mes"]
    + ajuste_segmento
    + np.random.normal(0, 260, n)
).clip(50).round(2)

clientes["compras_ultimo_anio"] = (
    2 * clientes["visitas_mes"]
    + np.random.poisson(5, n)
)

prob_abandono = (
    0.06
    + 0.22 * (clientes["satisfaccion"] < 3.5)
    + 0.09 * (clientes["visitas_mes"] <= 2)
)

clientes["abandono"] = np.where(
    np.random.rand(n) < prob_abandono,
    "Sí",
    "No"
)

clientes.head()

## 3.2 Incorporación de problemas de calidad

Se agregarán deliberadamente:

- valores faltantes;
- un duplicado;
- una edad sospechosa;
- valores extremos en gasto y visitas.

In [ ]:
clientes.loc[[7, 29, 85, 141, 220], "ingreso_mensual"] = np.nan
clientes.loc[[12, 44, 173, 250], "satisfaccion"] = np.nan
clientes.loc[[31, 112, 301], "region"] = np.nan

clientes.loc[5, "edad"] = 120
clientes.loc[18, "gasto_mensual"] = 15000
clientes.loc[55, "visitas_mes"] = 35

clientes = pd.concat(
    [clientes, clientes.iloc[[10]]],
    ignore_index=True
)

clientes.tail()

# 4. Inspección inicial de la base

## 4.1 Primeras observaciones

In [ ]:
clientes.head()

## 4.2 Últimas observaciones

In [ ]:
clientes.tail()

## 4.3 Dimensiones

La propiedad `shape` devuelve:

\[
(n,p),
\]

donde \(n\) es el número de filas y \(p\) es el número de columnas.

In [ ]:
clientes.shape

## 4.4 Nombres de las variables

In [ ]:
clientes.columns.tolist()

## 4.5 Tipos de datos

In [ ]:
clientes.dtypes

## 4.6 Información general

In [ ]:
clientes.info()

## 4.7 Cantidad de valores únicos

In [ ]:
clientes.nunique()

## 4.8 Selección de observaciones

In [ ]:
clientes.loc[
    0:4,
    ["cliente_id", "edad", "region", "ingreso_mensual"]
]

## 4.9 Clasificación de las variables

| Variable | Clasificación |
|---|---|
| cliente_id | Identificador |
| edad | Cuantitativa discreta |
| genero | Cualitativa nominal |
| region | Cualitativa nominal |
| segmento | Cualitativa ordinal |
| ingreso_mensual | Cuantitativa continua |
| gasto_mensual | Cuantitativa continua |
| visitas_mes | Cuantitativa discreta |
| satisfaccion | Cuantitativa ordinal |
| compras_ultimo_anio | Cuantitativa discreta |
| abandono | Cualitativa nominal binaria |

In [ ]:
variables_cualitativas = [
    "genero",
    "region",
    "segmento",
    "abandono"
]

variables_cuantitativas = [
    "edad",
    "ingreso_mensual",
    "gasto_mensual",
    "visitas_mes",
    "satisfaccion",
    "compras_ultimo_anio"
]

print("Cualitativas:", variables_cualitativas)
print("Cuantitativas:", variables_cuantitativas)

# 5. Calidad de los datos

## 5.1 Valores faltantes

Sea \(m_j\) el número de valores faltantes en la variable \(j\). El porcentaje de faltantes se calcula mediante:

\[
\text{Porcentaje de faltantes}_j
=
\frac{m_j}{n}\times 100.
\]

Un porcentaje pequeño no implica automáticamente que pueda ignorarse. Es necesario estudiar la causa de la ausencia.

In [ ]:
faltantes = clientes.isna().sum().to_frame("cantidad")

faltantes["porcentaje"] = (
    faltantes["cantidad"]
    / len(clientes)
    * 100
).round(2)

faltantes

In [ ]:
faltantes_visibles = faltantes[
    faltantes["cantidad"] > 0
]

plt.figure(figsize=(9, 5))
plt.bar(
    faltantes_visibles.index,
    faltantes_visibles["cantidad"]
)
plt.title("Cantidad de valores faltantes por variable")
plt.xlabel("Variable")
plt.ylabel("Cantidad")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

### Tipos conceptuales de datos faltantes

- **MCAR:** la ausencia no depende de variables observadas ni no observadas.
- **MAR:** la ausencia depende de variables observadas.
- **MNAR:** la ausencia depende del propio valor no observado.

En este notebook se usa una imputación sencilla por razones pedagógicas. En aplicaciones reales, el mecanismo de ausencia debe estudiarse.

## 5.2 Registros duplicados

In [ ]:
clientes.duplicated().sum()

In [ ]:
clientes[clientes.duplicated(keep=False)]

## 5.3 Valores imposibles o sospechosos

La revisión de mínimos y máximos permite detectar observaciones inconsistentes.

Por ejemplo:

- una edad de 120 años puede ser sospechosa;
- 35 visitas mensuales puede ser posible, pero extraordinario;
- un gasto de 15 000 puede ser un error o un cliente excepcional.

In [ ]:
clientes[variables_cuantitativas].agg(
    ["min", "max"]
)

In [ ]:
clientes[clientes["edad"] > 100]

In [ ]:
clientes[clientes["visitas_mes"] > 20]

In [ ]:
clientes[clientes["gasto_mensual"] > 10000]

## 5.4 Creación de una copia de trabajo

In [ ]:
df = clientes.copy()

## 5.5 Eliminación de duplicados

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
print("Nueva dimensión:", df.shape)

## 5.6 Imputación de faltantes

### Variables cuantitativas

Se utiliza la mediana:

\[
\tilde{x}
=
\begin{cases}
x_{\left(\frac{n+1}{2}\right)}, & n \text{ impar},\\[4pt]
\dfrac{x_{(n/2)}+x_{(n/2+1)}}{2}, & n \text{ par}.
\end{cases}
\]

La mediana es robusta frente a valores extremos.

### Variables cualitativas

Se utiliza la moda, es decir, la categoría con mayor frecuencia.

In [ ]:
df["ingreso_mensual"] = df["ingreso_mensual"].fillna(
    df["ingreso_mensual"].median()
)

df["satisfaccion"] = df["satisfaccion"].fillna(
    df["satisfaccion"].median()
)

df["region"] = df["region"].fillna(
    df["region"].mode()[0]
)

df.isna().sum()

## 5.7 Corrección de una edad sospechosa

In [ ]:
mediana_edad_valida = df.loc[
    df["edad"] <= 100,
    "edad"
].median()

df.loc[
    df["edad"] > 100,
    "edad"
] = mediana_edad_valida

df["edad"].max()

# 6. Estadística descriptiva

La estadística descriptiva resume y organiza los datos sin intentar generalizar más allá de la información observada.

Se divide en:

1. medidas de tendencia central;
2. medidas de dispersión;
3. medidas de posición;
4. medidas de forma;
5. tablas de frecuencia;
6. representaciones gráficas.

## 6.1 Resumen automático

In [ ]:
df[variables_cuantitativas].describe().T

# 7. Medidas de tendencia central

## 7.1 Media aritmética

Para una muestra \(x_1,\dots,x_n\), la media es:

$$
\bar{x}
=
\frac{1}{n}
\sum_{i=1}^{n}x_i.
$$

La media representa el punto de equilibrio de los datos.

### Propiedades

$$
\sum_{i=1}^{n}(x_i-\bar{x})=0.
$$

Además, la media minimiza la suma de cuadrados:

$$
\bar{x}
=
\arg\min_{a}
\sum_{i=1}^{n}(x_i-a)^2.
$$

La media es sensible a valores extremos.

In [ ]:
gasto = df["gasto_mensual"]

media_gasto = gasto.mean()

print(f"Media del gasto: {media_gasto:,.2f}")

## 7.2 Mediana

La mediana es el valor que divide los datos ordenados en dos partes con igual número de observaciones.

La mediana minimiza la suma de desviaciones absolutas:

$$
\tilde{x}
=
\arg\min_{a}
\sum_{i=1}^{n}|x_i-a|.
$$

Por esta razón es más robusta que la media.

In [ ]:
mediana_gasto = gasto.median()

print(f"Mediana del gasto: {mediana_gasto:,.2f}")

## 7.3 Moda

La moda es el valor o categoría con mayor frecuencia.

En variables cualitativas, la moda es una medida de tendencia central especialmente útil.

In [ ]:
moda_segmento = df["segmento"].mode()[0]

print("Segmento modal:", moda_segmento)

## 7.4 Comparación entre media y mediana

In [ ]:
comparacion_centro = df[
    variables_cuantitativas
].agg(
    ["mean", "median"]
).T

comparacion_centro["media_menos_mediana"] = (
    comparacion_centro["mean"]
    - comparacion_centro["median"]
)

comparacion_centro

### Interpretación

- Si media y mediana son parecidas, la distribución puede ser aproximadamente simétrica.
- Si la media es mucho mayor que la mediana, puede existir asimetría positiva.
- Si la media es mucho menor que la mediana, puede existir asimetría negativa.

# 8. Medidas de dispersión

Las medidas de dispersión cuantifican qué tan separados están los datos.

## 8.1 Rango

\[
R
=
x_{\max}
-
x_{\min}.
\]

El rango utiliza únicamente los valores extremos y, por tanto, es sensible a atípicos.

In [ ]:
rango_gasto = gasto.max() - gasto.min()
print(f"Rango: {rango_gasto:,.2f}")

## 8.2 Varianza poblacional

Si los datos representan toda la población:

\[
\sigma^2
=
\frac{1}{N}
\sum_{i=1}^{N}(x_i-\mu)^2.
\]

## 8.3 Varianza muestral

Si los datos representan una muestra:

\[
s^2
=
\frac{1}{n-1}
\sum_{i=1}^{n}(x_i-\bar{x})^2.
\]

El denominador \(n-1\) corresponde a la corrección de Bessel.

In [ ]:
varianza_gasto = gasto.var(ddof=1)
print(f"Varianza muestral: {varianza_gasto:,.2f}")

## 8.4 Desviación estándar

La desviación estándar es la raíz cuadrada de la varianza:

\[
s
=
\sqrt{
\frac{1}{n-1}
\sum_{i=1}^{n}(x_i-\bar{x})^2
}.
\]

Se expresa en las mismas unidades de la variable.

In [ ]:
desviacion_gasto = gasto.std(ddof=1)
print(f"Desviación estándar: {desviacion_gasto:,.2f}")

## 8.5 Rango intercuartílico

\[
IQR
=
Q_3-Q_1.
\]

El IQR mide la amplitud del 50 % central de los datos y es resistente a valores extremos.

In [ ]:
q1_gasto = gasto.quantile(0.25)
q3_gasto = gasto.quantile(0.75)
iqr_gasto = q3_gasto - q1_gasto

print(f"Q1: {q1_gasto:,.2f}")
print(f"Q3: {q3_gasto:,.2f}")
print(f"IQR: {iqr_gasto:,.2f}")

## 8.6 Coeficiente de variación

\[
CV
=
\frac{s}{\bar{x}}\times 100\%.
\]

El coeficiente de variación expresa la dispersión relativa respecto a la media.

Debe utilizarse con precaución cuando:

- la media es cercana a cero;
- la variable puede tomar valores negativos;
- las escalas no son comparables.

In [ ]:
cv_gasto = gasto.std() / gasto.mean() * 100
print(f"Coeficiente de variación: {cv_gasto:.2f}%")

## 8.7 Tabla descriptiva completa

In [ ]:
def resumen_descriptivo(serie):
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)

    return pd.Series({
        "n": serie.count(),
        "media": serie.mean(),
        "mediana": serie.median(),
        "desviacion": serie.std(),
        "varianza": serie.var(),
        "minimo": serie.min(),
        "q1": q1,
        "q3": q3,
        "maximo": serie.max(),
        "rango": serie.max() - serie.min(),
        "iqr": q3 - q1,
        "cv_porcentaje": serie.std() / serie.mean() * 100
    })

tabla_descriptiva = df[
    variables_cuantitativas
].apply(
    resumen_descriptivo
).T.round(2)

tabla_descriptiva

# 9. Medidas de posición

Un cuantil de orden \(p\), denotado \(Q(p)\), es un valor que deja aproximadamente una proporción \(p\) de observaciones por debajo.

Casos importantes:

\[
Q_1 = Q(0.25),
\qquad
Q_2 = Q(0.50),
\qquad
Q_3 = Q(0.75).
\]

El segundo cuartil coincide con la mediana.

In [ ]:
df["ingreso_mensual"].quantile(
    [0.10, 0.25, 0.50, 0.75, 0.90]
)

## 9.1 Interpretación de percentiles

Si el percentil 90 del ingreso es \(P_{90}\), entonces aproximadamente el 90 % de los clientes tiene un ingreso menor o igual que \(P_{90}\).

# 10. Tablas de frecuencias

Para una categoría \(c_j\):

## Frecuencia absoluta

\[
f_j
=
\#\{i:x_i=c_j\}.
\]

## Frecuencia relativa

\[
h_j
=
\frac{f_j}{n}.
\]

## Porcentaje

\[
p_j
=
100h_j.
\]

Debe cumplirse:

\[
\sum_j f_j=n,
\qquad
\sum_j h_j=1,
\qquad
\sum_j p_j=100.
\]

In [ ]:
frecuencia_segmento = df[
    "segmento"
].value_counts().to_frame(
    "frecuencia"
)

frecuencia_segmento["frecuencia_relativa"] = (
    frecuencia_segmento["frecuencia"]
    / len(df)
)

frecuencia_segmento["porcentaje"] = (
    frecuencia_segmento["frecuencia_relativa"]
    * 100
).round(2)

frecuencia_segmento

In [ ]:
print(
    "Suma de frecuencias:",
    frecuencia_segmento["frecuencia"].sum()
)

print(
    "Suma de frecuencias relativas:",
    frecuencia_segmento["frecuencia_relativa"].sum()
)

# 11. Análisis univariado de variables cualitativas

## 11.1 Gráfico de barras

In [ ]:
orden_segmentos = [
    "Básico",
    "Intermedio",
    "Premium"
]

plt.figure(figsize=(9, 5))

ax = sns.countplot(
    data=df,
    x="segmento",
    order=orden_segmentos
)

plt.title("Distribución de clientes por segmento")
plt.xlabel("Segmento")
plt.ylabel("Cantidad de clientes")

for contenedor in ax.containers:
    ax.bar_label(contenedor)

plt.tight_layout()
plt.show()

### Interpretación

El gráfico de barras permite comparar frecuencias entre categorías. La altura de cada barra representa el número de observaciones de la categoría correspondiente.

## 11.2 Gráfico horizontal de porcentajes

In [ ]:
porcentaje_region = (
    df["region"]
    .value_counts(normalize=True)
    .mul(100)
    .sort_values()
)

plt.figure(figsize=(9, 5))

plt.barh(
    porcentaje_region.index,
    porcentaje_region.values
)

plt.title("Porcentaje de clientes por región")
plt.xlabel("Porcentaje")
plt.ylabel("Región")

plt.tight_layout()
plt.show()

# 12. Análisis univariado de variables cuantitativas

## 12.1 Histograma

El histograma divide el rango de una variable en intervalos.

La frecuencia de la clase \(j\) es:

\[
f_j
=
\#\{x_i\in I_j\}.
\]

Un histograma ayuda a estudiar:

- forma;
- concentración;
- dispersión;
- asimetría;
- multimodalidad;
- posibles atípicos.

La elección del número de intervalos modifica la apariencia del gráfico.

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=df,
    x="ingreso_mensual",
    bins=25
)

plt.title("Distribución del ingreso mensual")
plt.xlabel("Ingreso mensual")
plt.ylabel("Frecuencia")

plt.tight_layout()
plt.show()

## 12.2 Regla de Freedman–Diaconis

Una forma de elegir el ancho de clase es:

\[
h
=
2\frac{IQR}{n^{1/3}}.
\]

Entonces, el número aproximado de intervalos puede calcularse como:

\[
k
\approx
\frac{x_{\max}-x_{\min}}{h}.
\]

In [ ]:
serie_ingreso = df["ingreso_mensual"]

iqr_ingreso = (
    serie_ingreso.quantile(0.75)
    - serie_ingreso.quantile(0.25)
)

h = 2 * iqr_ingreso / (len(serie_ingreso) ** (1/3))

k = int(
    np.ceil(
        (serie_ingreso.max() - serie_ingreso.min())
        / h
    )
)

print("Ancho recomendado:", round(h, 2))
print("Número aproximado de intervalos:", k)

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=df,
    x="ingreso_mensual",
    bins=k
)

plt.title("Histograma con regla de Freedman–Diaconis")
plt.tight_layout()
plt.show()

## 12.3 Estimación de densidad

La estimación de densidad por núcleos tiene la forma:

\[
\widehat{f}_h(x)
=
\frac{1}{nh}
\sum_{i=1}^{n}
K\left(
\frac{x-x_i}{h}
\right),
\]

donde:

- \(K\) es una función núcleo;
- \(h\) es el parámetro de suavizado.

La densidad es una representación suavizada de la distribución.

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=df,
    x="satisfaccion",
    bins=12,
    stat="density"
)

sns.kdeplot(
    data=df,
    x="satisfaccion",
    linewidth=2
)

plt.title("Histograma y densidad de la satisfacción")
plt.xlabel("Satisfacción")

plt.tight_layout()
plt.show()

## 12.4 Diagrama de caja

El boxplot se construye con:

- \(Q_1\);
- mediana;
- \(Q_3\);
- bigotes;
- posibles atípicos.

Los límites teóricos usuales son:

\[
L_I
=
Q_1-1.5IQR,
\]

\[
L_S
=
Q_3+1.5IQR.
\]

In [ ]:
plt.figure(figsize=(10, 4))

sns.boxplot(
    data=df,
    x="gasto_mensual"
)

plt.title("Diagrama de caja del gasto mensual")
plt.xlabel("Gasto mensual")

plt.tight_layout()
plt.show()

## 12.5 Gráfico de violín

In [ ]:
plt.figure(figsize=(8, 6))

sns.violinplot(
    data=df,
    y="ingreso_mensual",
    inner="quartile"
)

plt.title("Distribución del ingreso mensual")
plt.ylabel("Ingreso mensual")

plt.tight_layout()
plt.show()

# 13. Forma de una distribución

## 13.1 Asimetría

El coeficiente muestral de asimetría estudia la falta de simetría.

De forma conceptual:

\[
g_1
\propto
\frac{
\sum_{i=1}^{n}(x_i-\bar{x})^3
}{
s^3
}.
\]

Interpretación:

- \(g_1>0\): cola hacia la derecha;
- \(g_1<0\): cola hacia la izquierda;
- \(g_1\approx 0\): simetría aproximada.

In [ ]:
asimetria = df[
    variables_cuantitativas
].skew().sort_values(
    ascending=False
)

asimetria

## 13.2 Curtosis

La curtosis estudia el peso de las colas y la concentración de la distribución.

Una formulación basada en el cuarto momento es:

\[
\beta_2
=
\frac{
E[(X-\mu)^4]
}{
\sigma^4
}.
\]

Pandas reporta el exceso de curtosis, comparando con la distribución normal.

In [ ]:
curtosis = df[
    variables_cuantitativas
].kurt().sort_values(
    ascending=False
)

curtosis

## 13.3 Comparación visual

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5)
)

sns.histplot(
    data=df,
    x="ingreso_mensual",
    bins=25,
    ax=axes[0]
)

axes[0].axvline(
    df["ingreso_mensual"].mean(),
    linestyle="--",
    label="Media"
)

axes[0].axvline(
    df["ingreso_mensual"].median(),
    linestyle=":",
    label="Mediana"
)

axes[0].set_title("Ingreso mensual")
axes[0].legend()

sns.histplot(
    data=df,
    x="satisfaccion",
    bins=12,
    ax=axes[1]
)

axes[1].set_title("Satisfacción")

plt.tight_layout()
plt.show()

# 14. Detección de valores atípicos

## 14.1 Regla del IQR

Definimos:

\[
IQR=Q_3-Q_1.
\]

Los límites son:

\[
L_I=Q_1-1.5IQR,
\]

\[
L_S=Q_3+1.5IQR.
\]

Una observación fuera de estos límites se considera potencialmente atípica.

In [ ]:
def detectar_atipicos_iqr(serie):
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    mascara = (
        (serie < limite_inferior)
        | (serie > limite_superior)
    )

    return mascara, limite_inferior, limite_superior

In [ ]:
mascara_gasto, li_gasto, ls_gasto = (
    detectar_atipicos_iqr(
        df["gasto_mensual"]
    )
)

print(f"Límite inferior: {li_gasto:,.2f}")
print(f"Límite superior: {ls_gasto:,.2f}")
print("Número de atípicos:", mascara_gasto.sum())

In [ ]:
df.loc[
    mascara_gasto,
    [
        "cliente_id",
        "gasto_mensual",
        "ingreso_mensual",
        "segmento",
        "region"
    ]
].sort_values(
    "gasto_mensual",
    ascending=False
)

## 14.2 Puntaje estandarizado

El puntaje \(z\) se define como:

\[
z_i
=
\frac{x_i-\bar{x}}{s}.
\]

Una regla informal identifica como extremos los valores con:

\[
|z_i|>3.
\]

Esta regla funciona mejor cuando la distribución es aproximadamente simétrica.

In [ ]:
df["z_gasto"] = (
    df["gasto_mensual"]
    - df["gasto_mensual"].mean()
) / df["gasto_mensual"].std()

df.loc[
    df["z_gasto"].abs() > 3,
    [
        "cliente_id",
        "gasto_mensual",
        "z_gasto"
    ]
].sort_values(
    "z_gasto",
    ascending=False
)

### Interpretación responsable

Un valor atípico no es necesariamente un error. Puede representar:

- una observación extraordinaria;
- un cliente especial;
- un subgrupo diferente;
- un error de digitación;
- un evento raro pero real.

Eliminar atípicos sin justificación puede destruir información importante.

# 15. Análisis bivariado

El análisis bivariado estudia la relación entre dos variables.

La técnica depende de sus tipos:

| Variable 1 | Variable 2 | Herramientas |
|---|---|---|
| Cuantitativa | Cuantitativa | dispersión, covarianza, correlación |
| Cualitativa | Cuantitativa | boxplot, violín, resumen por grupos |
| Cualitativa | Cualitativa | tabla de contingencia, porcentajes |

## 15.1 Covarianza

Para dos variables \(X\) y \(Y\), la covarianza muestral es:

\[
s_{XY}
=
\frac{1}{n-1}
\sum_{i=1}^{n}
(x_i-\bar{x})(y_i-\bar{y}).
\]

Interpretación:

- \(s_{XY}>0\): tienden a aumentar conjuntamente;
- \(s_{XY}<0\): una tiende a aumentar cuando la otra disminuye;
- \(s_{XY}\approx 0\): asociación lineal débil.

La covarianza depende de las unidades de medida.

In [ ]:
covarianza = df[
    [
        "ingreso_mensual",
        "gasto_mensual"
    ]
].cov()

covarianza

## 15.2 Correlación de Pearson

\[
r_{XY}
=
\frac{s_{XY}}{s_Xs_Y}.
\]

El coeficiente cumple:

\[
-1\le r_{XY}\le 1.
\]

Interpretación general:

- \(r\approx 1\): asociación lineal positiva fuerte;
- \(r\approx -1\): asociación lineal negativa fuerte;
- \(r\approx 0\): asociación lineal débil.

La correlación no implica causalidad.

In [ ]:
correlacion_ingreso_gasto = df[
    [
        "ingreso_mensual",
        "gasto_mensual"
    ]
].corr()

correlacion_ingreso_gasto

## 15.3 Diagrama de dispersión

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="ingreso_mensual",
    y="gasto_mensual",
    alpha=0.65
)

plt.title("Relación entre ingreso y gasto mensual")
plt.xlabel("Ingreso mensual")
plt.ylabel("Gasto mensual")

plt.tight_layout()
plt.show()

## 15.4 Recta de tendencia

Una relación lineal puede resumirse mediante:

\[
\widehat{y}
=
b_0+b_1x.
\]

La pendiente \(b_1\) representa el cambio esperado en \(Y\) asociado con un incremento de una unidad en \(X\).

In [ ]:
plt.figure(figsize=(10, 6))

sns.regplot(
    data=df,
    x="ingreso_mensual",
    y="gasto_mensual",
    scatter_kws={
        "alpha": 0.40
    },
    line_kws={
        "linewidth": 2
    }
)

plt.title("Tendencia lineal entre ingreso y gasto")
plt.tight_layout()
plt.show()

## 15.5 Matriz de correlaciones

In [ ]:
matriz_correlacion = df[
    variables_cuantitativas
].corr()

matriz_correlacion.round(2)

In [ ]:
plt.figure(figsize=(11, 8))

sns.heatmap(
    matriz_correlacion,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    square=True
)

plt.title("Matriz de correlaciones")
plt.tight_layout()
plt.show()

# 16. Variable cualitativa frente a cuantitativa

Para comparar una variable cuantitativa entre grupos conviene estudiar:

- tamaño de cada grupo;
- media;
- mediana;
- dispersión;
- forma de la distribución;
- posibles atípicos.

In [ ]:
resumen_segmento = df.groupby(
    "segmento"
)["gasto_mensual"].agg(
    cantidad="count",
    media="mean",
    mediana="median",
    desviacion="std",
    minimo="min",
    q1=lambda x: x.quantile(0.25),
    q3=lambda x: x.quantile(0.75),
    maximo="max"
).reindex(
    orden_segmentos
)

resumen_segmento.round(2)

## 16.1 Boxplot por segmento

In [ ]:
plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df,
    x="segmento",
    y="gasto_mensual",
    order=orden_segmentos
)

plt.title("Gasto mensual según segmento")
plt.xlabel("Segmento")
plt.ylabel("Gasto mensual")

plt.tight_layout()
plt.show()

## 16.2 Violinplot por segmento

In [ ]:
plt.figure(figsize=(10, 6))

sns.violinplot(
    data=df,
    x="segmento",
    y="gasto_mensual",
    order=orden_segmentos,
    inner="quartile"
)

plt.title("Distribución del gasto por segmento")
plt.tight_layout()
plt.show()

## 16.3 Promedio por segmento

In [ ]:
promedio_segmento = (
    df.groupby(
        "segmento"
    )["gasto_mensual"]
    .mean()
    .reindex(
        orden_segmentos
    )
)

plt.figure(figsize=(9, 5))

ax = sns.barplot(
    x=promedio_segmento.index,
    y=promedio_segmento.values
)

plt.title("Gasto promedio por segmento")
plt.xlabel("Segmento")
plt.ylabel("Gasto promedio")

for contenedor in ax.containers:
    ax.bar_label(
        contenedor,
        fmt="%.0f"
    )

plt.tight_layout()
plt.show()

# 17. Relación entre variables cualitativas

## 17.1 Tabla de contingencia

Para dos variables cualitativas \(A\) y \(B\), una tabla de contingencia contiene frecuencias:

\[
n_{ij}
=
\#\{
\text{observaciones en categoría }i\text{ de }A
\text{ y categoría }j\text{ de }B
\}.
\]

In [ ]:
tabla_abandono_segmento = pd.crosstab(
    df["segmento"],
    df["abandono"]
)

tabla_abandono_segmento

## 17.2 Proporciones condicionadas por fila

\[
p_{ij}
=
\frac{n_{ij}}{n_{i+}},
\]

donde \(n_{i+}\) es el total de la fila \(i\).

Estas proporciones permiten comparar grupos con tamaños diferentes.

In [ ]:
porcentajes_abandono = pd.crosstab(
    df["segmento"],
    df["abandono"],
    normalize="index"
).mul(100).round(2)

porcentajes_abandono

## 17.3 Barras apiladas

In [ ]:
porcentajes_abandono.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 5)
)

plt.title("Proporción de abandono por segmento")
plt.xlabel("Segmento")
plt.ylabel("Porcentaje")
plt.xticks(rotation=0)
plt.legend(title="Abandono")

plt.tight_layout()
plt.show()

# 18. Visualización multivariada de patrones

## 18.1 Dispersión con color y estilo

In [ ]:
plt.figure(figsize=(11, 7))

sns.scatterplot(
    data=df,
    x="ingreso_mensual",
    y="gasto_mensual",
    hue="segmento",
    style="abandono",
    alpha=0.75
)

plt.title(
    "Ingreso y gasto según segmento y abandono"
)

plt.tight_layout()
plt.show()

## 18.2 Facetas

In [ ]:
grafico = sns.relplot(
    data=df,
    x="ingreso_mensual",
    y="gasto_mensual",
    col="segmento",
    hue="abandono",
    col_order=orden_segmentos,
    alpha=0.70,
    height=4,
    aspect=0.95
)

grafico.fig.suptitle(
    "Relación ingreso-gasto por segmento",
    y=1.05
)

plt.show()

## 18.3 Pairplot

In [ ]:
muestra_pairplot = df.sample(
    min(180, len(df)),
    random_state=42
)

sns.pairplot(
    muestra_pairplot,
    vars=[
        "edad",
        "ingreso_mensual",
        "gasto_mensual",
        "visitas_mes",
        "satisfaccion"
    ],
    hue="segmento",
    corner=True
)

plt.show()

## 18.4 Tabla dinámica

In [ ]:
tabla_region_segmento = df.pivot_table(
    index="region",
    columns="segmento",
    values="gasto_mensual",
    aggfunc="mean"
).reindex(
    columns=orden_segmentos
)

tabla_region_segmento.round(2)

## 18.5 Mapa de calor de una tabla dinámica

In [ ]:
plt.figure(figsize=(10, 6))

sns.heatmap(
    tabla_region_segmento,
    annot=True,
    fmt=".0f",
    cmap="YlGnBu"
)

plt.title("Gasto promedio por región y segmento")
plt.xlabel("Segmento")
plt.ylabel("Región")

plt.tight_layout()
plt.show()

# 19. Creación de variables derivadas

## 19.1 Grupos de edad

In [ ]:
intervalos_edad = [
    0,
    25,
    40,
    55,
    100
]

etiquetas_edad = [
    "18-25",
    "26-40",
    "41-55",
    "56 o más"
]

df["grupo_edad"] = pd.cut(
    df["edad"],
    bins=intervalos_edad,
    labels=etiquetas_edad
)

df[
    [
        "edad",
        "grupo_edad"
    ]
].head(10)

In [ ]:
gasto_por_edad = df.groupby(
    "grupo_edad",
    observed=False
)["gasto_mensual"].mean()

plt.figure(figsize=(9, 5))

ax = sns.barplot(
    x=gasto_por_edad.index,
    y=gasto_por_edad.values
)

plt.title("Gasto promedio por grupo de edad")
plt.xlabel("Grupo de edad")
plt.ylabel("Gasto promedio")

for contenedor in ax.containers:
    ax.bar_label(
        contenedor,
        fmt="%.0f"
    )

plt.tight_layout()
plt.show()

## 19.2 Transformación logarítmica

Para una variable positiva \(X\), una transformación común es:

\[
Y=\log(1+X).
\]

El término \(1\) permite incluir valores iguales a cero.

La transformación logarítmica:

- reduce asimetría positiva;
- comprime valores grandes;
- facilita comparaciones;
- puede hacer más visible una estructura aproximadamente lineal.

In [ ]:
df["log_ingreso"] = np.log1p(
    df["ingreso_mensual"]
)

df["log_gasto"] = np.log1p(
    df["gasto_mensual"]
)

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5)
)

sns.histplot(
    data=df,
    x="ingreso_mensual",
    bins=25,
    ax=axes[0]
)

axes[0].set_title(
    "Ingreso mensual original"
)

sns.histplot(
    data=df,
    x="log_ingreso",
    bins=25,
    ax=axes[1]
)

axes[1].set_title(
    "Logaritmo del ingreso"
)

plt.tight_layout()
plt.show()

# 20. Identificación de patrones de abandono

In [ ]:
resumen_abandono = df.groupby(
    "abandono"
).agg(
    cantidad=("cliente_id", "count"),
    satisfaccion_promedio=("satisfaccion", "mean"),
    visitas_promedio=("visitas_mes", "mean"),
    gasto_promedio=("gasto_mensual", "mean"),
    compras_promedio=("compras_ultimo_anio", "mean")
).round(2)

resumen_abandono

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5)
)

sns.boxplot(
    data=df,
    x="abandono",
    y="satisfaccion",
    ax=axes[0]
)

axes[0].set_title(
    "Satisfacción según abandono"
)

sns.boxplot(
    data=df,
    x="abandono",
    y="visitas_mes",
    ax=axes[1]
)

axes[1].set_title(
    "Visitas según abandono"
)

plt.tight_layout()
plt.show()

### Lectura esperada

Si quienes abandonan presentan menor satisfacción y menos visitas, se observa un patrón de asociación.

Sin embargo, este resultado no demuestra causalidad. Puede haber variables adicionales no observadas que expliquen la relación.

# 21. Función reutilizable para un EDA inicial

In [ ]:
def resumen_eda(dataframe):
    print("DIMENSIONES")
    print(dataframe.shape)

    print("\nTIPOS DE DATOS")
    print(dataframe.dtypes)

    print("\nVALORES FALTANTES")
    print(dataframe.isna().sum())

    print("\nDUPLICADOS")
    print(dataframe.duplicated().sum())

    print("\nRESUMEN NUMÉRICO")
    display(
        dataframe
        .select_dtypes(
            include="number"
        )
        .describe()
        .T
    )

    print("\nRESUMEN CUALITATIVO")
    display(
        dataframe
        .select_dtypes(
            exclude="number"
        )
        .describe()
        .T
    )

resumen_eda(df)

# 22. Principios para interpretar correctamente

## 22.1 Correlación no implica causalidad

Si dos variables se relacionan, no significa necesariamente que una cause la otra.

## 22.2 Un gráfico no reemplaza una conclusión

Toda visualización debe acompañarse de una interpretación.

## 22.3 La media no describe toda la distribución

También deben revisarse:

- mediana;
- cuartiles;
- dispersión;
- forma;
- valores atípicos.

## 22.4 Un atípico no debe eliminarse automáticamente

Debe investigarse su origen.

## 22.5 Comparar porcentajes puede ser más adecuado que comparar frecuencias

Esto ocurre especialmente cuando los grupos tienen tamaños diferentes.

## 22.6 Los resultados dependen de la calidad de los datos

Una base mal depurada puede producir conclusiones incorrectas.

# 23. Errores frecuentes en un EDA

1. No identificar la unidad de observación.
2. Tratar identificadores como magnitudes.
3. Analizar sin revisar faltantes.
4. Eliminar duplicados sin verificar su significado.
5. Eliminar todos los atípicos.
6. Usar gráficos inadecuados.
7. Ocultar escalas o unidades.
8. Confundir asociación con causalidad.
9. Sobreinterpretar correlaciones pequeñas.
10. Presentar tablas sin interpretación.

# 24. Ejercicios prácticos

## Ejercicio 1. Inspección inicial


Responda:

1. ¿Cuántas filas y columnas tiene `df`?
2. ¿Cuál es la unidad de observación?
3. ¿Cuántas variables cualitativas existen?
4. ¿Cuántas variables cuantitativas existen?
5. ¿Qué variable funciona como identificador?

In [ ]:
# Escriba aquí su solución

## Ejercicio 2. Calidad de datos


Compruebe:

1. valores faltantes;
2. duplicados;
3. edades menores de 18;
4. gastos superiores a 10 000;
5. visitas superiores a 20.

In [ ]:
# Escriba aquí su solución

## Ejercicio 3. Tendencia central


Para `ingreso_mensual`, calcule:

- media;
- mediana;
- moda;
- diferencia entre media y mediana.

Interprete el resultado.

In [ ]:
# Escriba aquí su solución

## Ejercicio 4. Dispersión


Para `gasto_mensual`, calcule:

- rango;
- varianza;
- desviación estándar;
- Q1;
- Q3;
- IQR;
- coeficiente de variación.

In [ ]:
# Escriba aquí su solución

## Ejercicio 5. Tabla de frecuencias


Construya una tabla de frecuencias absolutas, relativas y porcentuales para `region`.

In [ ]:
# Escriba aquí su solución

## Ejercicio 6. Distribución


Construya para `compras_ultimo_anio`:

1. un histograma;
2. una curva de densidad;
3. un boxplot.

Describa forma, centro, dispersión y posibles atípicos.

In [ ]:
# Escriba aquí su solución

## Ejercicio 7. Comparación entre grupos


Compare `ingreso_mensual` entre los segmentos Básico, Intermedio y Premium.

Use:

- tabla descriptiva;
- boxplot;
- gráfico de violín.

In [ ]:
# Escriba aquí su solución

## Ejercicio 8. Relación cuantitativa


Analice la relación entre `visitas_mes` y `compras_ultimo_anio`.

Use:

- covarianza;
- correlación;
- diagrama de dispersión;
- línea de tendencia.

In [ ]:
# Escriba aquí su solución

## Ejercicio 9. Relación cualitativa


Construya una tabla de contingencia entre `region` y `abandono`.

Calcule también porcentajes por región.

In [ ]:
# Escriba aquí su solución

## Ejercicio 10. Valores atípicos


Aplique la regla del IQR a `ingreso_mensual`.

Identifique los clientes potencialmente atípicos.

In [ ]:
# Escriba aquí su solución

## Ejercicio 11. Mapa de calor


Construya la matriz de correlaciones y responda:

1. ¿Cuál es la correlación positiva más alta?
2. ¿Qué relación parece débil?
3. ¿Qué advertencia debe hacerse antes de interpretar?

In [ ]:
# Escriba aquí su solución

## Ejercicio 12. Patrones de abandono


Compare a quienes abandonan con quienes no abandonan en:

- satisfacción;
- visitas;
- gasto;
- compras.

Construya una tabla y dos gráficos.

In [ ]:
# Escriba aquí su solución

# 25. Actividad integradora

## Caso: perfil y comportamiento de clientes

Realice un análisis exploratorio completo para responder:

1. ¿Cómo es el perfil general de los clientes?
2. ¿Qué segmento tiene mayor gasto promedio?
3. ¿Qué variables se asocian con el gasto?
4. ¿Qué diferencias existen entre quienes abandonan y quienes permanecen?
5. ¿En qué regiones se concentra el abandono?
6. ¿Qué observaciones atípicas merecen investigación?
7. ¿Qué decisiones de negocio podrían proponerse?

## Productos esperados

- identificación de la unidad de observación;
- clasificación de variables;
- tabla de calidad de datos;
- resumen estadístico;
- tablas de frecuencia;
- al menos seis visualizaciones;
- interpretación de cada gráfico;
- conclusiones;
- limitaciones.

In [ ]:
# Desarrolle aquí la actividad integradora

# 26. Soluciones orientativas

## Solución orientativa 1

In [ ]:
print("Dimensión:", df.shape)
print("Unidad: un cliente")
print(
    "Cualitativas:",
    len(df.select_dtypes(exclude="number").columns)
)
print(
    "Cuantitativas:",
    len(df.select_dtypes(include="number").columns)
)
print("Identificador: cliente_id")

## Solución orientativa 2

In [ ]:
print(df.isna().sum())
print("Duplicados:", df.duplicated().sum())
print("Edades < 18:", (df["edad"] < 18).sum())
print("Gasto > 10000:", (df["gasto_mensual"] > 10000).sum())
print("Visitas > 20:", (df["visitas_mes"] > 20).sum())

## Solución orientativa 3

In [ ]:
s = df["ingreso_mensual"]

print("Media:", s.mean())
print("Mediana:", s.median())
print("Moda:", s.mode().iloc[0])
print("Diferencia:", s.mean() - s.median())

## Solución orientativa 4

In [ ]:
s = df["gasto_mensual"]

q1 = s.quantile(0.25)
q3 = s.quantile(0.75)

print("Rango:", s.max() - s.min())
print("Varianza:", s.var())
print("Desviación:", s.std())
print("Q1:", q1)
print("Q3:", q3)
print("IQR:", q3 - q1)
print("CV:", s.std() / s.mean() * 100)

## Solución orientativa 5

In [ ]:
tabla = df["region"].value_counts().to_frame("frecuencia")
tabla["frecuencia_relativa"] = tabla["frecuencia"] / len(df)
tabla["porcentaje"] = tabla["frecuencia_relativa"] * 100
tabla

## Solución orientativa 6

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(data=df, x="compras_ultimo_anio", bins=20, stat="density")
sns.kdeplot(data=df, x="compras_ultimo_anio")
plt.show()

plt.figure(figsize=(9, 4))
sns.boxplot(data=df, x="compras_ultimo_anio")
plt.show()

## Solución orientativa 7

In [ ]:
resumen = df.groupby("segmento")["ingreso_mensual"].agg(
    ["count", "mean", "median", "std", "min", "max"]
).reindex(orden_segmentos)

display(resumen)

plt.figure(figsize=(9, 5))
sns.boxplot(
    data=df,
    x="segmento",
    y="ingreso_mensual",
    order=orden_segmentos
)
plt.show()

plt.figure(figsize=(9, 5))
sns.violinplot(
    data=df,
    x="segmento",
    y="ingreso_mensual",
    order=orden_segmentos
)
plt.show()

## Solución orientativa 8

In [ ]:
print(
    df[
        ["visitas_mes", "compras_ultimo_anio"]
    ].cov()
)

print(
    df[
        ["visitas_mes", "compras_ultimo_anio"]
    ].corr()
)

plt.figure(figsize=(9, 6))
sns.regplot(
    data=df,
    x="visitas_mes",
    y="compras_ultimo_anio"
)
plt.show()

## Solución orientativa 9

In [ ]:
tabla = pd.crosstab(
    df["region"],
    df["abandono"]
)

porcentajes = pd.crosstab(
    df["region"],
    df["abandono"],
    normalize="index"
).mul(100)

display(tabla)
display(porcentajes)

## Solución orientativa 10

In [ ]:
mascara, li, ls = detectar_atipicos_iqr(
    df["ingreso_mensual"]
)

print("Límite inferior:", li)
print("Límite superior:", ls)

df.loc[
    mascara,
    ["cliente_id", "ingreso_mensual"]
].sort_values(
    "ingreso_mensual",
    ascending=False
)

## Solución orientativa 11

In [ ]:
corr = df[variables_cuantitativas].corr()

plt.figure(figsize=(10, 7))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)
plt.show()

## Solución orientativa 12

In [ ]:
resumen = df.groupby("abandono").agg(
    satisfaccion_promedio=("satisfaccion", "mean"),
    visitas_promedio=("visitas_mes", "mean"),
    gasto_promedio=("gasto_mensual", "mean"),
    compras_promedio=("compras_ultimo_anio", "mean")
)

display(resumen)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot(
    data=df,
    x="abandono",
    y="satisfaccion",
    ax=axes[0]
)

sns.boxplot(
    data=df,
    x="abandono",
    y="visitas_mes",
    ax=axes[1]
)

plt.tight_layout()
plt.show()

# 27. Conclusiones

Un análisis exploratorio completo debe integrar:

\[
\text{comprensión}
+
\text{calidad}
+
\text{descripción}
+
\text{visualización}
+
\text{interpretación}.
\]

Las medidas numéricas y los gráficos se complementan:

- la media resume el centro;
- la mediana aporta robustez;
- la desviación mide dispersión;
- los cuantiles describen posición;
- la asimetría describe forma;
- el boxplot ayuda a detectar atípicos;
- la correlación resume asociación lineal;
- las tablas cruzadas comparan categorías;
- las visualizaciones multivariadas revelan patrones complejos.

El objetivo del EDA no es producir muchas salidas, sino transformar datos en evidencia comprensible y útil.